# LAB·J1 · The trace, read raw

**Hardware:** any machine. ~1.5 h.

This lab proves that a jitted function does not run your Python. It runs a recording of your Python, made once, on values that carry a shape and a dtype and no actual numbers. Once you can read that recording (the jaxpr) and predict where it breaks, tracing stops being mysterious and becomes something you can reason about line by line.

The rule for this lab, and every lab after it: predict before you run. Each exercise has an empty "your prediction" cell. Write down what you expect, in your own words, before you execute the cell that reveals the answer. Skipping the prediction skips the part that actually builds the skill.


In [ ]:
import jax
import jax.numpy as jnp

print(jax.__version__, jax.devices())


## Tracing a function you did not write

Take this small attention-style function. It calls a matmul, a transpose, and a softmax, three ops you already trust.

```python
def attend(q, k):
    return jax.nn.softmax(q @ k.T)
```

`jax.make_jaxpr` shows you the recording without compiling anything. Before you run the cell below, predict how many primitive operations `jax.nn.softmax` expands into once it is traced. The matmul and the transpose are easy to picture; count the steps softmax itself hides.


**Your prediction:**


In [ ]:
import jax
import jax.numpy as jnp

def attend(q, k):
    return jax.nn.softmax(q @ k.T)

print(jax.make_jaxpr(attend)(jnp.ones((4, 8)), jnp.ones((4, 8))))


## Reading the recording

Run the cell above on jax 0.4.38, CPU, and this is the jaxpr you get, verbatim:

```
{ lambda ; a:f32[4,8] b:f32[4,8]. let
    c:f32[8,4] = transpose[permutation=(1, 0)] b
    d:f32[4,4] = dot_general[
      dimension_numbers=(([1], [0]), ([], []))
      preferred_element_type=float32
    ] a c
    e:f32[4] = reduce_max[axes=(1,)] d
    f:f32[4] = max -inf e
    g:f32[4,1] = broadcast_in_dim[
      broadcast_dimensions=(0,)
      shape=(4, 1)
      sharding=None
    ] f
    h:f32[4,1] = stop_gradient g
    i:f32[4,4] = sub d h
    j:f32[4,4] = exp i
    k:f32[4] = reduce_sum[axes=(1,)] j
    l:f32[4,1] = broadcast_in_dim[
      broadcast_dimensions=(0,)
      shape=(4, 1)
      sharding=None
    ] k
    m:f32[4,4] = div j l
  in (m,) }
```

Two lines you wrote, twelve the machine remembers. `softmax` alone expands into seven of those primitives: a max reduction for numerical stability, the `stop_gradient` that keeps that stabilizing max out of the gradient, the subtract, the exp, the sum, the divide, and one of the two broadcasts. The other broadcast and the matmul belong to parts of `attend` you can already read directly: the matmul shows up as `dot_general` with its contraction spelled out as `dimension_numbers` rather than the `@` you wrote, and `.T` becomes an explicit `transpose[permutation=(1, 0)]`.

Go back to your prediction. Did you count the `stop_gradient`? Nothing about ordinary use of `jax.nn.softmax` suggests a gradient is involved at all. It is written into the primitive's own definition, and you only see it because you asked for the recording instead of trusting the API from the outside.


## Now trace your own

Write any small function of your own, two or three operations, and trace it the same way. A reduction or a `jnp.where` tends to produce more interesting equations than plain elementwise arithmetic. The stub below is a starting point; replace the body with something of your own before you run it.


In [ ]:
def my_function(x, y):
    # replace this body with a small computation of your own
    return jnp.sum(x * y)

print(jax.make_jaxpr(my_function)(jnp.ones(4), jnp.ones(4)))


## Annotate the equations

For each equation JAX printed, from your own trace and from `attend` above, write one line: what the primitive does, and which part of your source line produced it. This is the actual skill this chapter builds. Reading a jaxpr once is fine; reading one by habit is what lets you predict a recompile, a memory spike, or a gradient bug before you hit it.


**Your annotations:**


## The error you are meant to hit

Every trace replaces your arguments with tracers: values that carry a shape and a dtype and no actual numbers. That is fine for `x + y` or `x @ y`. It stops being fine the moment your code asks a traced value a yes-or-no question with Python's own `if`.

Predict what happens when you jit the function below and call it. Does it raise? If it raises, what do you expect the message to say, in your own words?

```python
@jax.jit
def sign_flip(x):
    if x.sum() > 0:
        return -x
    return x
```


**Your prediction:**


In [ ]:
import jax
import jax.numpy as jnp

@jax.jit
def sign_flip(x):
    if x.sum() > 0:
        return -x
    return x

sign_flip(jnp.array([1.0, 2.0, 3.0]))


## Reading the error

The call raises `TracerBoolConversionError`, and the message underneath it is exact: "Attempted boolean conversion of traced array with shape bool[]." `x.sum() > 0` produced a traced boolean, a value that will exist once real numbers flow through the function, and Python's `if` needs an answer right now, at trace time, before any numbers exist.

This is not a bug in your function. It is the trace doing exactly what it is supposed to do: recording one program that has to work for every future input, not deciding a branch for the one input it happened to see first. Chapter six on control flow gives you the fix, `lax.cond` and `jnp.where` among them; this lab only needs you to recognize the shape of the error on sight.


## Seeing values anyway

`print` inside a jitted function only ever prints a tracer, once, at trace time; it never shows you a real number, no matter how many times you call the function afterward. `jax.debug.print` is different: it stages a print into the compiled program itself, so it runs on every call, with the real values that call produced.

Predict what the cell below prints, and how many times you would need to call `step` again with a different array to see a second print.


**Your prediction:**


In [ ]:
import jax
import jax.numpy as jnp

@jax.jit
def step(x):
    jax.debug.print("x = {x}", x=x)   # runs every call, with real values
    return x * 2

step(jnp.arange(3.0))


## What just ran

The print fired with real numbers: the actual array `step` received, not a placeholder. Call `step` again with a different array and the print fires again, every time, because it is baked into the compiled program rather than executed once during tracing. That is the whole difference between Python's `print` and `jax.debug.print` inside jit: one belongs to trace time and runs once; the other belongs to every run and shows you what actually happened.


## Mark it run

You have traced a function you did not write, traced one you did, met the trace's one hard rule by breaking it on purpose, and printed a real value out of a compiled program. Go back to chapter 2 on the chapter page and tick LAB·J1 as run.
